# Condense Paderborn for transfer to the GPU server

The raw dataset is 9.7 GB of which this project uses one channel (`vibration_1`) out of seven, at a rate it immediately downsamples. This keeps only that channel, resampled 64 kHz -> 25.6 kHz as float32, for the WHOLE dataset (every bearing code and operating condition, not just what today's config reads -- the marginal cost is small and it means no future config change forces another download at server rates).

Result is ~1 GB instead of 9.7 GB. It also removes ~72 s of `scipy.io.loadmat` from every training run, which matters once a hyperparameter sweep repeats it dozens of times.

## 1. Reconstruct the loader code

In [ ]:
import base64, pathlib
WORKDIR = pathlib.Path('/kaggle/working')
SRC_FILES = {
    'src/domain_adaptation/__init__.py': (
        'IiIiWmVyby1zaG90IGZhdWx0IGRpYWdub3NpcyB2aWEgdW5zdXBlcnZpc2VkIGRvbWFpbiBhZGFwdGF0aW9uIChEQU5OKS4KClRoaXJkLCBpbmRlcGVuZGVudCBleHBlcmltZW50IGluIHRoaXMgcmVwbyAoc2VlIERPTUFJTl9BREFQVEFUSU9OX1BMQU4ubWQpOgpgdmlicmF0aW9uX2FkL2AgKHNpbmdsZS1zZW5zb3IgQ1dSVSBhdXRvZW5jb2RlcikgYW5kIGBoeWRyYXVsaWNfZnVzaW9uX2FkL2AKKG11bHRpLXNlbnNvciBaZU1BIGZ1c2lvbiBhdXRvZW5jb2RlcikgYXJlIGJvdGggdW50b3VjaGVkLiBUaGlzIHBhY2thZ2UKY29tYmluZXMgQ1dSVSAoYWxyZWFkeSBsb2NhbCwgYGRhdGEvcmF3L2ApIGFuZCBQYWRlcmJvcm4gVW5pdmVyc2l0eSdzIEtBdApiZWFyaW5nIGRhdGFzZXQgKGF0dGFjaGVkIGFzIGEgS2FnZ2xlIGlucHV0IGRhdGFzZXQsIG5ldmVyIGRvd25sb2FkZWQgaW4KZnVsbCkgdG8gdHJhaW4gYSBjbGFzc2lmaWVyIHRoYXQgdHJhbnNmZXJzIGFjcm9zcyBib3RoIGRhdGFzZXRzIGFuZCBhCmhlbGQtb3V0IG9wZXJhdGluZyBjb25kaXRpb24sIHVzaW5nIG9ubHkgdGhhdCBjb25kaXRpb24ncyBIRUFMVEhZIGRhdGEgdG8KYWRhcHQgLS0gbmV2ZXIgYSBzaW5nbGUgbGFiZWxlZCAob3IgZXZlbiB1bmxhYmVsZWQpIGZhdWx0IGV4YW1wbGUgZnJvbSB0aGUKdGFyZ2V0IC0tIHRoZW4gY2hlY2tzIHdoZXRoZXIgaXQgY2FuIGRpYWdub3NlIHJlYWwgdGFyZ2V0IGZhdWx0cyBpdCBuZXZlcgp0cmFpbmVkIG9uLgoiIiIK'
    ),
    'src/domain_adaptation/data/__init__.py': (
        ''
    ),
    'src/domain_adaptation/data/pu_bearing_codes.py': (
        'IiIiQ2Fub25pY2FsIGJlYXJpbmctY29kZSAtPiBmYXVsdCBtZXRhZGF0YSBmb3IgdGhlIFBhZGVyYm9ybiBVbml2ZXJzaXR5IC8KS0F0LURhdGFDZW50ZXIgYmVhcmluZyBkYXRhc2V0IChMZXNzbWVpZXIgZXQgYWwuIDIwMTYsICJDb25kaXRpb24gTW9uaXRvcmluZwpvZiBCZWFyaW5nIERhbWFnZSBpbiBFbGVjdHJvbWVjaGFuaWNhbCBEcml2ZSBTeXN0ZW1zIikuCgpWZXJpZmllZCBhZ2FpbnN0IHRoZSByZWFsIGRhdGFzZXQgb24gMjAyNi0wOS0xMCAoc2VlCkRPTUFJTl9BREFQVEFUSU9OX1BMQU4ubWQncyAiUmV2aXNpb25zIiBzZWN0aW9uKTogZG93bmxvYWRlZCAzIHJlYWwgZmlsZXMKb2ZmIHRoZSBhdHRhY2hlZCBLYWdnbGUgbWlycm9yIChgZGlwcGF0ZWwwMy9wYWRlcmJvcm4tZGJgKSBhbmQgcGFnZWQKdGhyb3VnaCBpdHMgZnVsbCBmaWxlIGxpc3RpbmcgdmlhIHRoZSBLYWdnbGUgQVBJIHRvIGNvbmZpcm0gZXZlcnkgY29kZQpiZWxvdyBleGlzdHMsIHdpdGggMjAgcnVucyB1bmRlciBlYWNoIG9mIHRoZSBkYXRhc2V0J3MgNCBvcGVyYXRpbmcKY29uZGl0aW9ucy4KCmB1bml2ZXJzYWxfbGFiZWxgIG1hcHMgaW50byB0aGUgU0FNRSAzLWNsYXNzIHNwYWNlIENXUlUgdXNlcyAoc2VlCmN3cnVfbG9hZGVyLlVOSVZFUlNBTF9MQUJFTFMpOiAwPWhlYWx0aHksIDE9aW5uZXJfcmFjZSwgMj1vdXRlcl9yYWNlLgpDb21iaW5lZC1kYW1hZ2UgY29kZXMgKEtCKiwgYm90aCByYWNlcyBkYW1hZ2VkIG9uIG9uZSBwaHlzaWNhbCBiZWFyaW5nKQpnZXQgdW5pdmVyc2FsX2xhYmVsPU5vbmUgLS0gdGhleSBkb24ndCBmaXQgYSBzaW5nbGUtZmF1bHQtdHlwZSBsYWJlbCBhbmQKYXJlIGV4Y2x1ZGVkIGZyb20gdGhpcyBwcm9qZWN0J3MgY2xhc3NpZmljYXRpb24gdGFzazsga2VwdCBoZXJlIG9ubHkgZm9yCnJlZmVyZW5jZS9jb21wbGV0ZW5lc3MuCiIiIgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgpIRUFMVEhZID0gWyJLMDAxIiwgIkswMDIiLCAiSzAwMyIsICJLMDA0IiwgIkswMDUiLCAiSzAwNiJdCkFSVElGSUNJQUxfT1VURVIgPSBbIktBMDEiLCAiS0EwMyIsICJLQTA1IiwgIktBMDYiLCAiS0EwNyIsICJLQTA4IiwgIktBMDkiXQpBUlRJRklDSUFMX0lOTkVSID0gWyJLSTAxIiwgIktJMDMiLCAiS0kwNSIsICJLSTA3IiwgIktJMDgiXQpSRUFMX09VVEVSID0gWyJLQTA0IiwgIktBMTUiLCAiS0ExNiIsICJLQTIyIiwgIktBMzAiXQpSRUFMX0lOTkVSID0gWyJLSTA0IiwgIktJMTQiLCAiS0kxNiIsICJLSTE3IiwgIktJMTgiLCAiS0kyMSJdClJFQUxfQ09NQklORUQgPSBbIktCMjMiLCAiS0IyNCIsICJLQjI3Il0KCgpkZWYgX2J1aWxkX3RhYmxlKCkgLT4gZGljdFtzdHIsIGRpY3RdOgogICAgdGFibGU6IGRpY3Rbc3RyLCBkaWN0XSA9IHt9CiAgICBmb3IgY29kZSBpbiBIRUFMVEhZOgogICAgICAgIHRhYmxlW2NvZGVdID0geyJmYXVsdF90eXBlIjogImhlYWx0aHkiLCAiZGFtYWdlX21vZGUiOiAibm9uZSIsICJ1bml2ZXJzYWxfbGFiZWwiOiAwfQogICAgZm9yIGNvZGUgaW4gQVJUSUZJQ0lBTF9PVVRFUjoKICAgICAgICB0YWJsZVtjb2RlXSA9IHsiZmF1bHRfdHlwZSI6ICJvdXRlcl9yYWNlIiwgImRhbWFnZV9tb2RlIjogImFydGlmaWNpYWwiLCAidW5pdmVyc2FsX2xhYmVsIjogMn0KICAgIGZvciBjb2RlIGluIEFSVElGSUNJQUxfSU5ORVI6CiAgICAgICAgdGFibGVbY29kZV0gPSB7ImZhdWx0X3R5cGUiOiAiaW5uZXJfcmFjZSIsICJkYW1hZ2VfbW9kZSI6ICJhcnRpZmljaWFsIiwgInVuaXZlcnNhbF9sYWJlbCI6IDF9CiAgICBmb3IgY29kZSBpbiBSRUFMX09VVEVSOgogICAgICAgIHRhYmxlW2NvZGVdID0geyJmYXVsdF90eXBlIjogIm91dGVyX3JhY2UiLCAiZGFtYWdlX21vZGUiOiAicmVhbCIsICJ1bml2ZXJzYWxfbGFiZWwiOiAyfQogICAgZm9yIGNvZGUgaW4gUkVBTF9JTk5FUjoKICAgICAgICB0YWJsZVtjb2RlXSA9IHsiZmF1bHRfdHlwZSI6ICJpbm5lcl9yYWNlIiwgImRhbWFnZV9tb2RlIjogInJlYWwiLCAidW5pdmVyc2FsX2xhYmVsIjogMX0KICAgIGZvciBjb2RlIGluIFJFQUxfQ09NQklORUQ6CiAgICAgICAgdGFibGVbY29kZV0gPSB7ImZhdWx0X3R5cGUiOiAiY29tYmluZWQiLCAiZGFtYWdlX21vZGUiOiAicmVhbCIsICJ1bml2ZXJzYWxfbGFiZWwiOiBOb25lfQogICAgcmV0dXJuIHRhYmxlCgoKUFVfQkVBUklOR19DT0RFUyA9IF9idWlsZF90YWJsZSgpCg=='
    ),
    'src/domain_adaptation/data/pu_loader.py': (
        'IiIiUGFyc2UgdGhlIFBhZGVyYm9ybiBVbml2ZXJzaXR5IChQVSAvIEtBdC1EYXRhQ2VudGVyKSBiZWFyaW5nIGRhdGFzZXQuCgpMYXlvdXQgKGNvbmZpcm1lZCAyMDI2LTA5LTEwIGJ5IGRvd25sb2FkaW5nIDMgcmVhbCBmaWxlcyBvZmYgdGhlIGF0dGFjaGVkCkthZ2dsZSBtaXJyb3IsIGBkaXBwYXRlbDAzL3BhZGVyYm9ybi1kYmAgLS0gc2VlIERPTUFJTl9BREFQVEFUSU9OX1BMQU4ubWQncwoiUmV2aXNpb25zIiBzZWN0aW9uKToKCiAgICA8cm9vdD4vPGJlYXJpbmdfY29kZT4vPGNvbmRpdGlvbj5fPGJlYXJpbmdfY29kZT5fPHJ1bj4ubWF0CgplLmcuIGBLMDAxL04wOV9NMDdfRjEwX0swMDFfMS5tYXRgLiBgY29uZGl0aW9uYCBlbmNvZGVzIHJvdGF0aW9uYWwgc3BlZWQKKE4wOT05MDBycG0sIE4xNT0xNTAwcnBtKSwgbG9hZCB0b3JxdWUgKE0wMT0wLjFObSwgTTA3PTAuN05tKSBhbmQgcmFkaWFsCmZvcmNlIChGMDQ9NDAwTiwgRjEwPTEwMDBOKSAtLSBzZWUgcHVfYmVhcmluZ19jb2Rlcy5weSBmb3Igd2hhdCBlYWNoCmBiZWFyaW5nX2NvZGVgIG1lYW5zIChoZWFsdGh5L2FydGlmaWNpYWwvcmVhbCwgaW5uZXIvb3V0ZXIgcmFjZSkuCgpFYWNoIGAubWF0YCBmaWxlIGlzIGEgTUFUTEFCIHN0cnVjdCAodG9wLWxldmVsIGtleSA9IHRoZSBmaWxlbmFtZSB3aXRoIHRoZQpleHRlbnNpb24gc3RyaXBwZWQpIHdpdGggYSBgWWAgZmllbGQ6IGEgbGVuZ3RoLTcgbGlzdCBvZiBwZXItY2hhbm5lbApkaWN0cywgZWFjaCBgeydOYW1lJzogc3RyLCAnRGF0YSc6IDFEIGFycmF5fWAuIENoYW5uZWxzIGFuZCB0aGVpcgphcHByb3hpbWF0ZSBuYXRpdmUgcmF0ZXMgKGZyb20gdGhlIHNhbWUgaW5zcGVjdGlvbik6IGZvcmNlL3NwZWVkL3RvcnF1ZQpAIH40a0h6LCBwaGFzZV9jdXJyZW50XzEvMiBAIH42NGtIeiwgdGVtcF8yX2JlYXJpbmdfbW9kdWxlIEAgfjEuMjVIeiAoNQpzYW1wbGVzIHBlciA0cyBydW4pLCB2aWJyYXRpb25fMSBAIH42NGtIeiAtLSBvbmx5IGB2aWJyYXRpb25fMWAgaXMgdXNlZApoZXJlLCBtYXRjaGluZyBDV1JVJ3Mgc2luZ2xlLWFjY2VsZXJvbWV0ZXItY2hhbm5lbCBzY29wZS4gUmF3IHJ1biBsZW5ndGhzCnZhcnkgYnkgYSBmZXcgaHVuZHJlZCBzYW1wbGVzIGFyb3VuZCB0aGUgbm9taW5hbCA0cy8yNTYwMDAgc2FtcGxlcyAoYSBEQVEKcXVpcmssIG5vdCBhIGJ1ZykgLS0gbm90aGluZyBkb3duc3RyZWFtIGFzc3VtZXMgYW4gZXhhY3QgbGVuZ3RoLgoiIiIKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IHJlCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAoKaW1wb3J0IG51bXB5IGFzIG5wCmltcG9ydCBwYW5kYXMgYXMgcGQKaW1wb3J0IHNjaXB5LmlvCgpmcm9tIGRvbWFpbl9hZGFwdGF0aW9uLmRhdGEucHVfYmVhcmluZ19jb2RlcyBpbXBvcnQgUFVfQkVBUklOR19DT0RFUwoKRklMRU5BTUVfUkUgPSByZS5jb21waWxlKHIiXig/UDxjb25kaXRpb24+TlxkezJ9X01cZHsyfV9GXGR7Mn0pXyg/UDxjb2RlPltBLVpdK1xkKylfKD9QPHJ1bj5cZCspXC5tYXQkIikKCgpkZWYgZmluZF9wdV9yb290KGxvY2FsX2hpbnQ6IHN0ciB8IFBhdGggfCBOb25lID0gTm9uZSkgLT4gUGF0aDoKICAgICIiIkxvY2F0ZSB0aGUgUFUgZGF0YXNldCdzIHJvb3QgZGlyZWN0b3J5ICh0aGUgZm9sZGVyIHRoYXQgZGlyZWN0bHkKICAgIGNvbnRhaW5zIEswMDEvLCBLMDAyLywgLi4uIHN1YmZvbGRlcnMpLgoKICAgIE9uIEthZ2dsZSwgc2VhcmNoZXMgL2thZ2dsZS9pbnB1dCBmb3IgYSBmb2xkZXIgY29udGFpbmluZyBhIEswMDEvCiAgICBzdWJmb2xkZXIgLS0gYnkgY29udGVudCwgbm90IGEgZml4ZWQgcGF0aCwgYmVjYXVzZSBhdHRhY2hlZC1kYXRhc2V0CiAgICBtb3VudCBwYXRocyBoYXZlIHByb3ZlbiBpbmNvbnNpc3RlbnQgaW4gdGhpcyBwcm9qZWN0IGJlZm9yZSAoc2VlCiAgICB2aWJyYXRpb25fYWQncyBub3Rlcy8wN19kdmNfYW5kX2thZ2dsZS5tZCkuIEZhbGxzIGJhY2sgdG8KICAgIGBsb2NhbF9oaW50YCAoY29uZmlncy9kb21haW5fYWRhcHRhdGlvbi55YW1sJ3MgcGF0aHMucHVfcm9vdF9sb2NhbCkgZm9yCiAgICBsb2NhbCBkZXZlbG9wbWVudCBhZ2FpbnN0IGEgc21hbGwgaGFuZC1idWlsdCBzYW1wbGUuCiAgICAiIiIKICAgIGthZ2dsZV9pbnB1dCA9IFBhdGgoIi9rYWdnbGUvaW5wdXQiKQogICAgaWYga2FnZ2xlX2lucHV0LmlzX2RpcigpOgogICAgICAgIGNhbmRpZGF0ZXMgPSBzb3J0ZWQocC5wYXJlbnQgZm9yIHAgaW4ga2FnZ2xlX2lucHV0LnJnbG9iKCJLMDAxIikgaWYgcC5pc19kaXIoKSkKICAgICAgICBpZiBjYW5kaWRhdGVzOgogICAgICAgICAgICByZXR1cm4gY2FuZGlkYXRlc1swXQogICAgICAgIHJhaXNlIEZpbGVOb3RGb3VuZEVycm9yKAogICAgICAgICAgICBmIk5vIEswMDEvIGZvbGRlciBmb3VuZCB1bmRlciB7a2FnZ2xlX2lucHV0fSAtLSBpcyB0aGUgUFUgZGF0YXNldCAiCiAgICAgICAgICAgICJhdHRhY2hlZD8gKHNlZSBub3RlYm9va3MvZG9tYWluX2FkYXB0YXRpb24va2VybmVsLW1ldGFkYXRhLmpzb24ncyBkYXRhc2V0X3NvdXJjZXMpIgogICAgICAgICkKICAgIGlmIGxvY2FsX2hpbnQgaXMgbm90IE5vbmUgYW5kIFBhdGgobG9jYWxfaGludCkuaXNfZGlyKCk6CiAgICAgICAgcmV0dXJuIFBhdGgobG9jYWxfaGludCkKICAgIHJhaXNlIEZpbGVOb3RGb3VuZEVycm9yKAogICAgICAgICJOb3QgcnVubmluZyBvbiBLYWdnbGUgKC9rYWdnbGUvaW5wdXQgbWlzc2luZykgYW5kIG5vIHVzYWJsZSBsb2NhbCAiCiAgICAgICAgZiJwdV9yb290ICh7bG9jYWxfaGludCFyfSkgLS0gcGFzcyAtLXB1LXJvb3QgZXhwbGljaXRseS4iCiAgICApCgoKY2xhc3MgVW5yZWFkYWJsZU1hdEZpbGUoUnVudGltZUVycm9yKToKICAgICIiIkEgUFUgLm1hdCBmaWxlIHNjaXB5IGNhbm5vdCBwYXJzZS4gUmFpc2VkIGluc3RlYWQgb2YgbGV0dGluZyBzY2lweSdzCiAgICBiYXJlIGBUeXBlRXJyb3I6IEV4cGVjdGluZyBtYXRyaXggaGVyZWAgZXNjYXBlIHdpdGggbm8gaW5kaWNhdGlvbiBvZgogICAgV0hJQ0ggb2YgfjE0MDAgZmlsZXMgd2FzIGF0IGZhdWx0IC0tIHdoaWNoIGlzIGV4YWN0bHkgaG93IGtlcm5lbCB2NQogICAgZGllZCBhZnRlciAzIG1pbnV0ZXMgd2l0aCBhbiB1bmF0dHJpYnV0YWJsZSB0cmFjZWJhY2suCiAgICAiIiIKCgpkZWYgbG9hZF92aWJyYXRpb25fc2lnbmFsKG1hdF9wYXRoOiBzdHIgfCBQYXRoLCBjaGFubmVsOiBzdHIgPSAidmlicmF0aW9uXzEiKSAtPiBucC5uZGFycmF5OgogICAgIiIiRXh0cmFjdCBvbmUgY2hhbm5lbCdzIDFEIGFycmF5IGZyb20gYSBQVSAubWF0IGZpbGUuIiIiCiAgICB0cnk6CiAgICAgICAgZGF0YSA9IHNjaXB5LmlvLmxvYWRtYXQoc3RyKG1hdF9wYXRoKSwgc2ltcGxpZnlfY2VsbHM9VHJ1ZSkKICAgIGV4Y2VwdCBFeGNlcHRpb24gYXMgZXhjOiAgIyBzY2lweSByYWlzZXMgYmFyZSBUeXBlRXJyb3IvVmFsdWVFcnJvciB3aXRoIG5vIGZpbGUgY29udGV4dAogICAgICAgIHJhaXNlIFVucmVhZGFibGVNYXRGaWxlKGYie21hdF9wYXRofTogc2NpcHkgY291bGQgbm90IHBhcnNlIHRoaXMgLm1hdCAoe3R5cGUoZXhjKS5fX25hbWVfX306IHtleGN9KSIpIGZyb20gZXhjCiAgICBrZXkgPSBuZXh0KGsgZm9yIGsgaW4gZGF0YSBpZiBub3Qgay5zdGFydHN3aXRoKCJfXyIpKQogICAgY2hhbm5lbHMgPSBkYXRhW2tleV1bIlkiXQogICAgZm9yIGNoIGluIGNoYW5uZWxzOgogICAgICAgIGlmIGNoWyJOYW1lIl0gPT0gY2hhbm5lbDoKICAgICAgICAgICAgcmV0dXJuIG5wLmFzYXJyYXkoY2hbIkRhdGEiXSwgZHR5cGU9bnAuZmxvYXQ2NCkucmF2ZWwoKQogICAgYXZhaWxhYmxlID0gW2NoWyJOYW1lIl0gZm9yIGNoIGluIGNoYW5uZWxzXQogICAgcmFpc2UgS2V5RXJyb3IoZiJ7bWF0X3BhdGh9OiBjaGFubmVsIHtjaGFubmVsIXJ9IG5vdCBmb3VuZCAoaGFzOiB7YXZhaWxhYmxlfSkiKQoKCmRlZiBidWlsZF9wdV9jYXRhbG9nKHJvb3RfZGlyOiBzdHIgfCBQYXRoLCBjb2RlczogbGlzdFtzdHJdLCBuYXRpdmVfZnNfaHo6IGludCA9IDY0MDAwKSAtPiBwZC5EYXRhRnJhbWU6CiAgICAiIiJPbmUgcm93IHBlciBgPGNvZGU+Lzxjb25kaXRpb24+Xzxjb2RlPl88cnVuPi5tYXRgIGZpbGUgZm91bmQgdW5kZXIKICAgIGByb290X2RpcmAsIGZvciBldmVyeSBgY29kZWAgaW4gYGNvZGVzYCAoc2tpcHMgdGhlIHBlci1jb2RlIC5wZGYgZG9jcwogICAgYW5kIGFueXRoaW5nIHRoYXQgZG9lc24ndCBtYXRjaCB0aGUgbmFtaW5nIGNvbnZlbnRpb24gb3IgaGFzIG5vCiAgICB1bml2ZXJzYWwtbGFiZWwgbWFwcGluZywgZS5nLiBjb21iaW5lZC1kYW1hZ2UgY29kZXMgLS0gc2VlCiAgICBwdV9iZWFyaW5nX2NvZGVzLnB5KS4gUm93IG9yZGVyIGlzIGRldGVybWluaXN0aWMgKHNvcnRlZCBnbG9iKSwgbm90CiAgICBkaXNrIG9yZGVyLgoKICAgIGBuYXRpdmVfZnNfaHpgIGlzIGEgcGFyYW1ldGVyIChub3QgaGFyZGNvZGVkKSBzbyB0ZXN0cyBjYW4gZXhlcmNpc2UKICAgIHRoaXMgYWdhaW5zdCBzbWFsbCBzeW50aGV0aWMgc2lnbmFscyBhdCBhIGxvd2VyIHJhdGUgLS0gcHJvZHVjdGlvbgogICAgY2FsbGVycyBwYXNzIGNvbmZpZy5wdS5uYXRpdmVfZnNfaHogKDY0MDAwIGZvciB0aGUgcmVhbCBkYXRhc2V0KS4KCiAgICBSZXR1cm5zIGNvbHVtbnM6IGZpbGVwYXRoLCBjb2RlLCBjb25kaXRpb24sIHJ1biwgZmF1bHRfdHlwZSwKICAgIGRhbWFnZV9tb2RlLCB1bml2ZXJzYWxfbGFiZWwsIGZzX25hdGl2ZV9oeiwgZG9tYWluLgogICAgIiIiCiAgICByb290X2RpciA9IFBhdGgocm9vdF9kaXIpCiAgICByb3dzID0gW10KICAgIGZvciBjb2RlIGluIGNvZGVzOgogICAgICAgIGNvZGVfZGlyID0gcm9vdF9kaXIgLyBjb2RlCiAgICAgICAgaWYgbm90IGNvZGVfZGlyLmlzX2RpcigpOgogICAgICAgICAgICByYWlzZSBGaWxlTm90Rm91bmRFcnJvcihmIlBVIGJlYXJpbmcgY29kZSBmb2xkZXIgbm90IGZvdW5kOiB7Y29kZV9kaXJ9IikKICAgICAgICBtZXRhID0gUFVfQkVBUklOR19DT0RFUy5nZXQoY29kZSkKICAgICAgICBpZiBtZXRhIGlzIE5vbmU6CiAgICAgICAgICAgIHJhaXNlIEtleUVycm9yKGYiVW5rbm93biBQVSBiZWFyaW5nIGNvZGUge2NvZGUhcn0gLS0gbm90IGluIHB1X2JlYXJpbmdfY29kZXMuUFVfQkVBUklOR19DT0RFUyIpCiAgICAgICAgaWYgbWV0YVsidW5pdmVyc2FsX2xhYmVsIl0gaXMgTm9uZToKICAgICAgICAgICAgY29udGludWUgICMgY29tYmluZWQtZGFtYWdlIGNvZGUsIGV4Y2x1ZGVkIGZyb20gdGhpcyB0YXNrIChzZWUgbW9kdWxlIGRvY3N0cmluZykKCiAgICAgICAgZm9yIG1hdF9wYXRoIGluIHNvcnRlZChjb2RlX2Rpci5nbG9iKCIqLm1hdCIpKToKICAgICAgICAgICAgbSA9IEZJTEVOQU1FX1JFLm1hdGNoKG1hdF9wYXRoLm5hbWUpCiAgICAgICAgICAgIGlmIG5vdCBtIG9yIG0uZ3JvdXAoImNvZGUiKSAhPSBjb2RlOgogICAgICAgICAgICAgICAgY29udGludWUKICAgICAgICAgICAgcm93cy5hcHBlbmQoCiAgICAgICAgICAgICAgICB7CiAgICAgICAgICAgICAgICAgICAgImZpbGVwYXRoIjogc3RyKG1hdF9wYXRoKSwKICAgICAgICAgICAgICAgICAgICAiY29kZSI6IGNvZGUsCiAgICAgICAgICAgICAgICAgICAgImNvbmRpdGlvbiI6IG0uZ3JvdXAoImNvbmRpdGlvbiIpLAogICAgICAgICAgICAgICAgICAgICJydW4iOiBpbnQobS5ncm91cCgicnVuIikpLAogICAgICAgICAgICAgICAgICAgICJmYXVsdF90eXBlIjogbWV0YVsiZmF1bHRfdHlwZSJdLAogICAgICAgICAgICAgICAgICAgICJkYW1hZ2VfbW9kZSI6IG1ldGFbImRhbWFnZV9tb2RlIl0sCiAgICAgICAgICAgICAgICAgICAgInVuaXZlcnNhbF9sYWJlbCI6IG1ldGFbInVuaXZlcnNhbF9sYWJlbCJdLAogICAgICAgICAgICAgICAgICAgICJmc19uYXRpdmVfaHoiOiBuYXRpdmVfZnNfaHosCiAgICAgICAgICAgICAgICAgICAgImRvbWFpbiI6ICJwdSIsCiAgICAgICAgICAgICAgICB9CiAgICAgICAgICAgICkKICAgIGlmIG5vdCByb3dzOgogICAgICAgIHJhaXNlIEZpbGVOb3RGb3VuZEVycm9yKGYiTm8gbWF0Y2hpbmcgUFUgLm1hdCBmaWxlcyBmb3VuZCB1bmRlciB7cm9vdF9kaXJ9IGZvciBjb2RlcyB7Y29kZXN9IikKICAgIHJldHVybiBwZC5EYXRhRnJhbWUocm93cykKCiMgLS0tIGNvbmRlbnNlZCAoInByZXBhcmVkIikgc3RvcmUgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tCiMgcHJlcGFyZV9wdS5weSB0dXJucyB0aGUgOS43IEdCIHJhdyBkYXRhc2V0IGludG8gfjEgR0Igb2YgcGVyLWNvZGUgLm5wegojIGFyY2hpdmVzLiBFdmVyeXRoaW5nIGJlbG93IHJlYWRzIHRoYXQgZm9ybS4gRGV0ZWN0aW9uIGlzIGF1dG9tYXRpYywgc28gdGhlCiMgc2FtZSAtLXB1LXJvb3Qgd29ya3Mgd2hldGhlciBpdCBwb2ludHMgYXQgcmF3IC5tYXQgZm9sZGVycyBvciBhIGNvbmRlbnNlZAojIHN0b3JlIC0tIG5vIGNvbmZpZyBmbGFnIHRvIGZvcmdldCB0byBmbGlwLCBhbmQgbm8gc2Vjb25kIGNvZGUgcGF0aCB0aHJvdWdoCiMgdGhlIHJlc3Qgb2YgdGhlIHBpcGVsaW5lLgoKCmRlZiBpc19wcmVwYXJlZF9zdG9yZShyb290OiBzdHIgfCBQYXRoKSAtPiBib29sOgogICAgIiIiQSBjb25kZW5zZWQgc3RvcmUgaXMgaWRlbnRpZmllZCBieSBpdHMgbWFuaWZlc3Q7IHJhdyBQVSBoYXMgbm9uZS4iIiIKICAgIHJldHVybiAoUGF0aChyb290KSAvICJtYW5pZmVzdC5jc3YiKS5pc19maWxlKCkKCgpjbGFzcyBQcmVwYXJlZFBVU3RvcmU6CiAgICAiIiJSZWFkcyB0aGUgcGVyLWNvZGUgYXJjaGl2ZXMgd3JpdHRlbiBieSBwcmVwYXJlX3B1LnB5LgoKICAgIG51bXB5IGxvYWRzIC5ucHogbWVtYmVycyBsYXppbHksIHNvIGhvbGRpbmcgb25lIG9wZW4gaGFuZGxlIHBlciBiZWFyaW5nCiAgICBjb2RlIGFuZCBpbmRleGluZyBpbnRvIGl0IGF2b2lkcyByZS1yZWFkaW5nIGEgfjMzIE1CIGFyY2hpdmUgZm9yIGV2ZXJ5CiAgICBvbmUgb2YgaXRzIDgwIHJ1bnMuCiAgICAiIiIKCiAgICBkZWYgX19pbml0X18oc2VsZiwgcm9vdDogc3RyIHwgUGF0aCk6CiAgICAgICAgc2VsZi5yb290ID0gUGF0aChyb290KQogICAgICAgIG1hbmlmZXN0X3BhdGggPSBzZWxmLnJvb3QgLyAibWFuaWZlc3QuY3N2IgogICAgICAgIGlmIG5vdCBtYW5pZmVzdF9wYXRoLmlzX2ZpbGUoKToKICAgICAgICAgICAgcmFpc2UgRmlsZU5vdEZvdW5kRXJyb3IoZiJ7c2VsZi5yb290fSBpcyBub3QgYSBwcmVwYXJlZCBzdG9yZSAobm8gbWFuaWZlc3QuY3N2KSIpCiAgICAgICAgc2VsZi5tYW5pZmVzdCA9IHBkLnJlYWRfY3N2KG1hbmlmZXN0X3BhdGgpCiAgICAgICAgc2VsZi5faGFuZGxlczogZGljdFtzdHIsIG9iamVjdF0gPSB7fQoKICAgIGRlZiBfaGFuZGxlKHNlbGYsIGNvZGU6IHN0cik6CiAgICAgICAgaWYgY29kZSBub3QgaW4gc2VsZi5faGFuZGxlczoKICAgICAgICAgICAgcGF0aCA9IHNlbGYucm9vdCAvIGYie2NvZGV9Lm5weiIKICAgICAgICAgICAgaWYgbm90IHBhdGguaXNfZmlsZSgpOgogICAgICAgICAgICAgICAgcmFpc2UgRmlsZU5vdEZvdW5kRXJyb3IoZiJ7cGF0aH0gbWlzc2luZyAtLSBtYW5pZmVzdCBsaXN0cyB7Y29kZX0gYnV0IHRoZSBhcmNoaXZlIGlzIGFic2VudCIpCiAgICAgICAgICAgIHNlbGYuX2hhbmRsZXNbY29kZV0gPSBucC5sb2FkKHBhdGgpCiAgICAgICAgcmV0dXJuIHNlbGYuX2hhbmRsZXNbY29kZV0KCiAgICBkZWYgbG9hZChzZWxmLCBjb2RlOiBzdHIsIGNvbmRpdGlvbjogc3RyLCBydW46IGludCkgLT4gbnAubmRhcnJheToKICAgICAgICBrZXkgPSBmIntjb25kaXRpb259X3tydW59IgogICAgICAgIGhhbmRsZSA9IHNlbGYuX2hhbmRsZShjb2RlKQogICAgICAgIGlmIGtleSBub3QgaW4gaGFuZGxlOgogICAgICAgICAgICByYWlzZSBLZXlFcnJvcihmIntjb2RlfS5ucHogaGFzIG5vIHJ1biB7a2V5IXJ9IikKICAgICAgICByZXR1cm4gbnAuYXNhcnJheShoYW5kbGVba2V5XSwgZHR5cGU9bnAuZmxvYXQ2NCkKCiAgICBkZWYgY2F0YWxvZyhzZWxmLCBjb2RlczogbGlzdFtzdHJdKSAtPiBwZC5EYXRhRnJhbWU6CiAgICAgICAgIiIiU2FtZSBjb2x1bW5zIGJ1aWxkX3B1X2NhdGFsb2cgcmV0dXJucywgc28gZG93bnN0cmVhbSBjb2RlIGlzCiAgICAgICAgaWRlbnRpY2FsIGZvciBlaXRoZXIgc3RvcmFnZSBmb3JtLiIiIgogICAgICAgIHdhbnRlZCA9IHNlbGYubWFuaWZlc3Rbc2VsZi5tYW5pZmVzdC5jb2RlLmlzaW4oY29kZXMpXS5jb3B5KCkKICAgICAgICBtaXNzaW5nID0gc29ydGVkKHNldChjb2RlcykgLSBzZXQod2FudGVkLmNvZGUudW5pcXVlKCkpKQogICAgICAgIGlmIG1pc3Npbmc6CiAgICAgICAgICAgIHJhaXNlIEZpbGVOb3RGb3VuZEVycm9yKGYiUHJlcGFyZWQgc3RvcmUge3NlbGYucm9vdH0gaGFzIG5vIGRhdGEgZm9yIGNvZGVzIHttaXNzaW5nfSIpCiAgICAgICAgd2FudGVkID0gd2FudGVkW3dhbnRlZC51bml2ZXJzYWxfbGFiZWwubm90bmEoKV0KICAgICAgICB3YW50ZWRbInVuaXZlcnNhbF9sYWJlbCJdID0gd2FudGVkLnVuaXZlcnNhbF9sYWJlbC5hc3R5cGUoaW50KQogICAgICAgIHdhbnRlZFsiZmlsZXBhdGgiXSA9ICIiICAgICAgICAgICMgc2lnbmFscyBsaXZlIGluc2lkZSB0aGUgYXJjaGl2ZXMsIG5vdCBhcyBmaWxlcwogICAgICAgIHdhbnRlZFsiZnNfbmF0aXZlX2h6Il0gPSB3YW50ZWRbImZzX2h6Il0KICAgICAgICB3YW50ZWRbImRvbWFpbiJdID0gInB1IgogICAgICAgIGNvbHMgPSBbImZpbGVwYXRoIiwgImNvZGUiLCAiY29uZGl0aW9uIiwgInJ1biIsICJmYXVsdF90eXBlIiwKICAgICAgICAgICAgICAgICJkYW1hZ2VfbW9kZSIsICJ1bml2ZXJzYWxfbGFiZWwiLCAiZnNfbmF0aXZlX2h6IiwgImRvbWFpbiJdCiAgICAgICAgcmV0dXJuIHdhbnRlZFtjb2xzXS5yZXNldF9pbmRleChkcm9wPVRydWUpCg=='
    ),
    'src/domain_adaptation/data/transforms.py': (
        'IiIiQ3Jvc3MtZG9tYWluIHNhbXBsZS1yYXRlIGFsaWdubWVudC4KCkNXUlUncyBEcml2ZS1FbmQgc2lnbmFsIGlzIGFscmVhZHkgMTJrSHo7IFBhZGVyYm9ybidzIHZpYnJhdGlvbl8xIGNoYW5uZWwKaXMgNjRrSHouIEEgQ05OIHRyYWluZWQgb24gbG9nLVNURlQtbWFnbml0dWRlIGltYWdlcyBuZWVkcyBwaXhlbHMgdGhhdAptZWFuIHRoZSBzYW1lIHBoeXNpY2FsIGZyZXF1ZW5jeSByYW5nZSBpbiBldmVyeSBkb21haW4sIHNvIGV2ZXJ5IHNpZ25hbCBpcwpyZXNhbXBsZWQgdG8gb25lIGNvbW1vbiByYXRlIChgY29uZmlnLmNvbW1vbi5mc19oemAsIGRlZmF1bHQgMTIwMDApIEJFRk9SRQp3aW5kb3dpbmcvU1RGVCAtLSBzZWUgRE9NQUlOX0FEQVBUQVRJT05fUExBTi5tZCdzICJSZXZpc2lvbnMiIHNlY3Rpb24gZm9yCndoeSAxMmtIeiAoQ1dSVSdzIG5hdGl2ZSByYXRlKSB3YXMgY2hvc2VuIG92ZXIgdXAgc2FtcGxpbmcgZXZlcnl0aGluZyB0bwo0OGtIeiBhcyB0aGUgb3JpZ2luYWwgcGxhbiBwcm9wb3NlZC4KIiIiCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmZyb20gZnJhY3Rpb25zIGltcG9ydCBGcmFjdGlvbgoKaW1wb3J0IG51bXB5IGFzIG5wCmZyb20gc2NpcHkuc2lnbmFsIGltcG9ydCByZXNhbXBsZV9wb2x5CgoKZGVmIHJlc2FtcGxlX3RvX3JhdGUoc2lnbmFsOiBucC5uZGFycmF5LCBuYXRpdmVfZnNfaHo6IGludCwgdGFyZ2V0X2ZzX2h6OiBpbnQpIC0+IG5wLm5kYXJyYXk6CiAgICAiIiJQb2x5cGhhc2UgcmVzYW1wbGUgKHByb3BlciBhbnRpLWFsaWFzaW5nIGZpbHRlciwgdW5saWtlIGEgbmFpdmUKICAgIEZGVC1iaW4gcmVzYW1wbGUpIGZyb20gbmF0aXZlX2ZzX2h6IHRvIHRhcmdldF9mc19oei4KCiAgICBOby1vcHMgKHJldHVybnMgYHNpZ25hbGAgdW5jaGFuZ2VkLCBub3QgZXZlbiBjb3BpZWQpIHdoZW4gdGhlIHJhdGVzCiAgICBhbHJlYWR5IG1hdGNoIC0tIGUuZy4gQ1dSVSBhdCAxMmtIeiB3aGVuIHRhcmdldF9mc19oej0xMjAwMCAtLSBzbyBkYXRhCiAgICB0aGF0IGRvZXNuJ3QgbmVlZCByZXNhbXBsaW5nIG5ldmVyIHBpY2tzIHVwIGZpbHRlciBhcnRpZmFjdHMgZnJvbSBvbmUuCiAgICAiIiIKICAgIGlmIG5hdGl2ZV9mc19oeiA9PSB0YXJnZXRfZnNfaHo6CiAgICAgICAgcmV0dXJuIHNpZ25hbAogICAgcmF0aW8gPSBGcmFjdGlvbih0YXJnZXRfZnNfaHosIG5hdGl2ZV9mc19oeikubGltaXRfZGVub21pbmF0b3IoMTAwMCkKICAgIHJldHVybiByZXNhbXBsZV9wb2x5KHNpZ25hbCwgdXA9cmF0aW8ubnVtZXJhdG9yLCBkb3duPXJhdGlvLmRlbm9taW5hdG9yKS5hc3R5cGUoc2lnbmFsLmR0eXBlKQo='
    ),
    'src/domain_adaptation/data/prepare_pu.py': (
        'IiIiQ29uZGVuc2UgdGhlIFBhZGVyYm9ybiBkYXRhc2V0IGludG8gYSBjb21wYWN0LCB0cmFuc2Zlci1mcmllbmRseSBmb3JtLgoKUnVucyBPTiBLQUdHTEUsIHdoZXJlIHRoZSA5LjcgR0IgZGF0YXNldCBpcyBhbHJlYWR5IG1vdW50ZWQgYW5kIGNvbXB1dGUgaXMKZnJlZS4gV3JpdGVzIG9uZSBgLm5wemAgcGVyIGJlYXJpbmcgY29kZSBob2xkaW5nIGV2ZXJ5IHJ1bidzIHZpYnJhdGlvbgpzaWduYWwsIHJlc2FtcGxlZCBhbmQgc3RvcmVkIGFzIGZsb2F0MzIuIFRoZSByZXN1bHQgaXMgfjEgR0IgZm9yIHRoZSBlbnRpcmUKZGF0YXNldCwgd2hpY2ggaXMgd2hhdCBhY3R1YWxseSBnZXRzIG1vdmVkIHRvIHRoZSBHUFUgc2VydmVyLgoKV2h5IHRoaXMgZXhpc3RzIHJhdGhlciB0aGFuIGNvcHlpbmcgdGhlIHJhdyBkYXRhc2V0OgoKKiAqKlNldmVuIGNoYW5uZWxzIGluLCBvbmUgY2hhbm5lbCBvdXQuKiogRWFjaCBgLm1hdGAgaXMgfjguNyBNQiBjYXJyeWluZwogIGZvcmNlLCB0d28gcGhhc2UgY3VycmVudHMsIHNwZWVkLCB0ZW1wZXJhdHVyZSwgdG9ycXVlIGFuZCB2aWJyYXRpb24uIFRoaXMKICBwcm9qZWN0IHVzZXMgYHZpYnJhdGlvbl8xYCBhbG9uZSAtLSBldmVyeXRoaW5nIGVsc2UgaXMgYnl0ZXMgd2Ugd291bGQKICBwYXkgdG8gdHJhbnNmZXIsIHN0b3JlLCBhbmQgdGhlbiBuZXZlciByZWFkLgoqICoqNjQga0h6IGluLCAyNS42IGtIeiBvdXQuKiogVGhlIHBpcGVsaW5lIHJlc2FtcGxlcyBQVSBkb3duIGJlZm9yZSBkb2luZwogIGFueXRoaW5nIHdpdGggaXQsIHNvIHNoaXBwaW5nIHRoZSBvcmlnaW5hbCByYXRlIG1lYW5zIHBheWluZyB0byBtb3ZlCiAgc2FtcGxlcyB0aGF0IGdldCBkaXNjYXJkZWQgb24gYXJyaXZhbC4KKiAqKmBzY2lweS5pby5sb2FkbWF0YCBpcyBzbG93LioqIEJ1aWxkaW5nIHRoZSBkYXRhc2V0IGZyb20gMTQyMCByYXcgYC5tYXRgCiAgZmlsZXMgY29zdCB+NzIgcyBvZiBldmVyeSBLYWdnbGUgcnVuLiBSZWFkaW5nIHByZS1leHRyYWN0ZWQgYXJyYXlzIGlzIGEKICBmYXN0IGBucC5sb2FkYCwgd2hpY2ggbWF0dGVycyBvbmNlIGEgaHlwZXJwYXJhbWV0ZXIgc3dlZXAgcnVucyBpdCBkb3plbnMKICBvZiB0aW1lcy4KKiAqKlJlbnRlZCBjb21wdXRlIGlzIGJpbGxlZCBieSB0aGUgbWludXRlLioqIEEgMSBHQiB0cmFuc2ZlciBpbnN0ZWFkIG9mIGEKICA5LjcgR0Igb25lIGlzIHRoZSBkaWZmZXJlbmNlIHBlciBzZXJ2ZXIgcmVidWlsZC4KCkRlbGliZXJhdGVseSBjb252ZXJ0cyB0aGUgV0hPTEUgZGF0YXNldCwgbm90IGp1c3QgdGhlIHN1YnNldCB0aGUgY3VycmVudApjb25maWcgcmVhZHMuIFRoZSBtYXJnaW5hbCBjb3N0IGlzIHNtYWxsICh+MSBHQiB2cyB+MC43IEdCKSBhbmQgaXQgcmVtb3ZlcwphbnkgY2hhbmNlIG9mIGEgZnV0dXJlIGNvbmZpZyBjaGFuZ2UgLS0gYSBmb3VydGggc291cmNlIGNvbmRpdGlvbiwgbW9yZQpiZWFyaW5nIGNvZGVzLCB0aGUgY29tYmluZWQtZGFtYWdlIEtCKiB1bml0cyAtLSBmb3JjaW5nIGFub3RoZXIgZG93bmxvYWQgYXQKc2VydmVyIHJhdGVzLgoKMjUuNiBrSHogaXMgYSBjbGVhbiAyLzUgc3RlcCBkb3duIGZyb20gNjQga0h6IGFuZCBzaXRzIGFib3ZlIHRoZSAxMiBrSHogdGhlCmNyb3NzLWRhdGFzZXQgcGlwZWxpbmUgY3VycmVudGx5IHVzZXMgKENXUlUncyBERTEyIGZpbGVzIGNhcCB0aGUgKmNvbW1vbioKcmF0ZSBhdCAxMiBrSHogcmVnYXJkbGVzcyksIGxlYXZpbmcgaGVhZHJvb20gZm9yIFBVLW9ubHkgYW5kIG9yZGVyLWRvbWFpbgpleHBlcmltZW50cyB3aXRob3V0IGEgc2Vjb25kIHRyaXAuCgpVc2FnZSAob24gS2FnZ2xlKToKICAgIHB5dGhvbiAtbSBkb21haW5fYWRhcHRhdGlvbi5kYXRhLnByZXBhcmVfcHUgLS1wdS1yb290IDxtb3VudD4gLS1vdXQgcHJlcGFyZWRfcHUKIiIiCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCBhcmdwYXJzZQppbXBvcnQgcmUKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCgppbXBvcnQgbnVtcHkgYXMgbnAKaW1wb3J0IHBhbmRhcyBhcyBwZAoKZnJvbSBkb21haW5fYWRhcHRhdGlvbi5kYXRhLnB1X2JlYXJpbmdfY29kZXMgaW1wb3J0IFBVX0JFQVJJTkdfQ09ERVMKZnJvbSBkb21haW5fYWRhcHRhdGlvbi5kYXRhLnB1X2xvYWRlciBpbXBvcnQgRklMRU5BTUVfUkUsIFVucmVhZGFibGVNYXRGaWxlLCBsb2FkX3ZpYnJhdGlvbl9zaWduYWwKZnJvbSBkb21haW5fYWRhcHRhdGlvbi5kYXRhLnRyYW5zZm9ybXMgaW1wb3J0IHJlc2FtcGxlX3RvX3JhdGUKCk5BVElWRV9GU19IWiA9IDY0MDAwClBSRVBBUkVEX0ZTX0haID0gMjU2MDAgICMgNjQwMDAgKiAyLzUgLS0gZXhhY3QgcmF0aW9uYWwgc3RlcCwgbm8gcmVzYW1wbGluZyBlcnJvcgojIE5vbWluYWwgc2hhZnQgc3BlZWQgcGVyIFBVIG9wZXJhdGluZy1jb25kaXRpb24gY29kZSwgbmVlZGVkIGZvciBvcmRlcgojIHRyYWNraW5nIChkYXRhL29yZGVyX3RyYWNraW5nLnB5KS4gTjE1ID0gMTUwMCBycG0sIE4wOSA9IDkwMCBycG0uCkNPTkRJVElPTl9SUE0gPSB7Ik4xNSI6IDE1MDAsICJOMDkiOiA5MDB9CgoKZGVmIHJwbV9mb3JfY29uZGl0aW9uKGNvbmRpdGlvbjogc3RyKSAtPiBpbnQ6CiAgICAiIiJgTjA5X00wN19GMTBgIC0+IDkwMC4gU3BlZWQgaXMgdGhlIGZpcnN0IGZpZWxkOyB0b3JxdWUgYW5kIHJhZGlhbAogICAgZm9yY2UgKE0vRikgZG8gbm90IGNoYW5nZSBzaGFmdCByYXRlLiIiIgogICAgcHJlZml4ID0gY29uZGl0aW9uLnNwbGl0KCJfIilbMF0KICAgIGlmIHByZWZpeCBub3QgaW4gQ09ORElUSU9OX1JQTToKICAgICAgICByYWlzZSBLZXlFcnJvcihmIlVua25vd24gc3BlZWQgY29kZSB7cHJlZml4IXJ9IGluIGNvbmRpdGlvbiB7Y29uZGl0aW9uIXJ9IikKICAgIHJldHVybiBDT05ESVRJT05fUlBNW3ByZWZpeF0KCgpkZWYgcHJlcGFyZV9jb2RlKGNvZGVfZGlyOiBQYXRoLCBvdXRfZGlyOiBQYXRoLCBmc19vdXQ6IGludCA9IFBSRVBBUkVEX0ZTX0haKSAtPiBkaWN0OgogICAgIiIiQ29uZGVuc2UgZXZlcnkgcnVuIG9mIG9uZSBiZWFyaW5nIGNvZGUgaW50byBhIHNpbmdsZSAubnB6LgoKICAgIEFycmF5cyBhcmUga2V5ZWQgYDxjb25kaXRpb24+XzxydW4+YDsgYW4gdW5yZWFkYWJsZSBzb3VyY2UgZmlsZSBpcwogICAgc2tpcHBlZCBhbmQgY291bnRlZCByYXRoZXIgdGhhbiBhYm9ydGluZywgdGhlIHNhbWUgcG9zdHVyZQogICAgZGF0YS9kYXRhc2V0LnB5IHRha2VzIChvbmUgY29ycnVwdCBmaWxlIGluIHRoaXMgS2FnZ2xlIG1pcnJvciwKICAgIEtBMDgvTjE1X00wMV9GMTBfS0EwOF8yLm1hdCwgb25jZSBraWxsZWQgYW4gZW50aXJlIHJ1bikuCiAgICAiIiIKICAgIGNvZGUgPSBjb2RlX2Rpci5uYW1lCiAgICBhcnJheXMsIHJvd3MsIHVucmVhZGFibGUgPSB7fSwgW10sIFtdCiAgICBmb3IgbWF0X3BhdGggaW4gc29ydGVkKGNvZGVfZGlyLmdsb2IoIioubWF0IikpOgogICAgICAgIG0gPSBGSUxFTkFNRV9SRS5tYXRjaChtYXRfcGF0aC5uYW1lKQogICAgICAgIGlmIG5vdCBtIG9yIG0uZ3JvdXAoImNvZGUiKSAhPSBjb2RlOgogICAgICAgICAgICBjb250aW51ZQogICAgICAgIGNvbmRpdGlvbiwgcnVuID0gbS5ncm91cCgiY29uZGl0aW9uIiksIGludChtLmdyb3VwKCJydW4iKSkKICAgICAgICB0cnk6CiAgICAgICAgICAgIHNpZ25hbCA9IGxvYWRfdmlicmF0aW9uX3NpZ25hbChtYXRfcGF0aCkKICAgICAgICBleGNlcHQgVW5yZWFkYWJsZU1hdEZpbGUgYXMgZXhjOgogICAgICAgICAgICB1bnJlYWRhYmxlLmFwcGVuZChtYXRfcGF0aC5uYW1lKQogICAgICAgICAgICBwcmludChmIiAgICBTS0lQIHVucmVhZGFibGU6IHtleGN9IikKICAgICAgICAgICAgY29udGludWUKICAgICAgICByZXNhbXBsZWQgPSByZXNhbXBsZV90b19yYXRlKHNpZ25hbCwgTkFUSVZFX0ZTX0haLCBmc19vdXQpLmFzdHlwZShucC5mbG9hdDMyKQogICAgICAgIGFycmF5c1tmIntjb25kaXRpb259X3tydW59Il0gPSByZXNhbXBsZWQKICAgICAgICByb3dzLmFwcGVuZCgKICAgICAgICAgICAgewogICAgICAgICAgICAgICAgImNvZGUiOiBjb2RlLCAiY29uZGl0aW9uIjogY29uZGl0aW9uLCAicnVuIjogcnVuLAogICAgICAgICAgICAgICAgInJwbSI6IHJwbV9mb3JfY29uZGl0aW9uKGNvbmRpdGlvbiksICJmc19oeiI6IGZzX291dCwKICAgICAgICAgICAgICAgICJuX3NhbXBsZXMiOiBsZW4ocmVzYW1wbGVkKSwKICAgICAgICAgICAgICAgICJmYXVsdF90eXBlIjogUFVfQkVBUklOR19DT0RFU1tjb2RlXVsiZmF1bHRfdHlwZSJdLAogICAgICAgICAgICAgICAgImRhbWFnZV9tb2RlIjogUFVfQkVBUklOR19DT0RFU1tjb2RlXVsiZGFtYWdlX21vZGUiXSwKICAgICAgICAgICAgICAgICJ1bml2ZXJzYWxfbGFiZWwiOiBQVV9CRUFSSU5HX0NPREVTW2NvZGVdWyJ1bml2ZXJzYWxfbGFiZWwiXSwKICAgICAgICAgICAgfQogICAgICAgICkKCiAgICBvdXRfZGlyLm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKICAgIG91dF9wYXRoID0gb3V0X2RpciAvIGYie2NvZGV9Lm5weiIKICAgIG5wLnNhdmV6X2NvbXByZXNzZWQob3V0X3BhdGgsICoqYXJyYXlzKQogICAgbWIgPSBvdXRfcGF0aC5zdGF0KCkuc3Rfc2l6ZSAvIDEwMjQqKjIKICAgIHByaW50KGYiICB7Y29kZX06IHtsZW4oYXJyYXlzKX0gcnVucyAtPiB7b3V0X3BhdGgubmFtZX0gKHttYjouMWZ9IE1CKSIKICAgICAgICAgICsgKGYiICBbe2xlbih1bnJlYWRhYmxlKX0gdW5yZWFkYWJsZV0iIGlmIHVucmVhZGFibGUgZWxzZSAiIikpCiAgICByZXR1cm4geyJyb3dzIjogcm93cywgInVucmVhZGFibGUiOiB1bnJlYWRhYmxlLCAibWIiOiBtYn0KCgpkZWYgbWFpbigpIC0+IE5vbmU6CiAgICBwYXJzZXIgPSBhcmdwYXJzZS5Bcmd1bWVudFBhcnNlcihkZXNjcmlwdGlvbj1fX2RvY19fKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1wdS1yb290IiwgcmVxdWlyZWQ9VHJ1ZSwgaGVscD0iTW91bnRlZCBQYWRlcmJvcm4gZGF0YXNldCByb290LiIpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLW91dCIsIHJlcXVpcmVkPVRydWUsIGhlbHA9IkRpcmVjdG9yeSBmb3IgdGhlIGNvbmRlbnNlZCAubnB6IGZpbGVzLiIpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLWZzIiwgdHlwZT1pbnQsIGRlZmF1bHQ9UFJFUEFSRURfRlNfSFopCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLWNvZGVzIiwgbmFyZ3M9IioiLCBkZWZhdWx0PU5vbmUsCiAgICAgICAgICAgICAgICAgICAgICAgIGhlbHA9IkJlYXJpbmcgY29kZXMgdG8gY29udmVydCAoZGVmYXVsdDogZXZlcnkgY29kZSBmb3VuZCkuIikKICAgIGFyZ3MgPSBwYXJzZXIucGFyc2VfYXJncygpCgogICAgcm9vdCwgb3V0X2RpciA9IFBhdGgoYXJncy5wdV9yb290KSwgUGF0aChhcmdzLm91dCkKICAgIGNvZGVzID0gYXJncy5jb2RlcyBvciBzb3J0ZWQoCiAgICAgICAgZC5uYW1lIGZvciBkIGluIHJvb3QuaXRlcmRpcigpIGlmIGQuaXNfZGlyKCkgYW5kIHJlLmZ1bGxtYXRjaChyIktbQUJJXT9cZCsiLCBkLm5hbWUpCiAgICApCiAgICBwcmludChmIkNvbmRlbnNpbmcge2xlbihjb2Rlcyl9IGJlYXJpbmcgY29kZXMgZnJvbSB7cm9vdH0gYXQge2FyZ3MuZnN9IEh6XG4iKQoKICAgIGFsbF9yb3dzLCBhbGxfdW5yZWFkYWJsZSwgdG90YWxfbWIgPSBbXSwgW10sIDAuMAogICAgZm9yIGNvZGUgaW4gY29kZXM6CiAgICAgICAgcmVzdWx0ID0gcHJlcGFyZV9jb2RlKHJvb3QgLyBjb2RlLCBvdXRfZGlyLCBhcmdzLmZzKQogICAgICAgIGFsbF9yb3dzICs9IHJlc3VsdFsicm93cyJdCiAgICAgICAgYWxsX3VucmVhZGFibGUgKz0gcmVzdWx0WyJ1bnJlYWRhYmxlIl0KICAgICAgICB0b3RhbF9tYiArPSByZXN1bHRbIm1iIl0KCiAgICBtYW5pZmVzdCA9IHBkLkRhdGFGcmFtZShhbGxfcm93cykKICAgIG1hbmlmZXN0LnRvX2NzdihvdXRfZGlyIC8gIm1hbmlmZXN0LmNzdiIsIGluZGV4PUZhbHNlKQogICAgcHJpbnQoZiJcbntsZW4obWFuaWZlc3QpfSBydW5zIGFjcm9zcyB7bWFuaWZlc3QuY29kZS5udW5pcXVlKCl9IGNvZGVzLCB7dG90YWxfbWI6LjBmfSBNQiB0b3RhbCIpCiAgICBwcmludChmIm1hbmlmZXN0IC0+IHtvdXRfZGlyIC8gJ21hbmlmZXN0LmNzdid9IikKICAgIGlmIGFsbF91bnJlYWRhYmxlOgogICAgICAgIHByaW50KGYie2xlbihhbGxfdW5yZWFkYWJsZSl9IHVucmVhZGFibGUgc291cmNlIGZpbGVzIHNraXBwZWQ6IHthbGxfdW5yZWFkYWJsZVs6NV19IikKCgppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgogICAgbWFpbigpCg=='
    ),
}
for relpath, b64 in SRC_FILES.items():
    p = WORKDIR / relpath
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_bytes(base64.b64decode(b64))
print(f'wrote {len(SRC_FILES)} source files')

## 2. Locate the attached dataset

In [ ]:
import sys, os
sys.path.insert(0, str(WORKDIR / 'src'))
os.environ['PYTHONPATH'] = str(WORKDIR / 'src')
os.chdir(WORKDIR)
from domain_adaptation.data.pu_loader import find_pu_root
PU_ROOT = find_pu_root()
codes = sorted(d.name for d in PU_ROOT.iterdir() if d.is_dir())
print(f'PU_ROOT = {PU_ROOT}')
print(f'{len(codes)} bearing codes: {codes}')

## 3. Condense
Every code, every condition, every run. Unreadable source files are named and skipped rather than aborting -- this mirror has at least one (`KA08/N15_M01_F10_KA08_2.mat`), and it previously killed a whole run.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'domain_adaptation.data.prepare_pu',
                '--pu-root', str(PU_ROOT), '--out', 'prepared_pu'], check=True)

## 4. Confirm what will be transferred

In [ ]:
import pandas as pd
out = WORKDIR / 'prepared_pu'
files = sorted(out.glob('*.npz'))
total_mb = sum(f.stat().st_size for f in files) / 1024**2
manifest = pd.read_csv(out / 'manifest.csv')
print(f'{len(files)} archives, {total_mb:.0f} MB total '
      f'(raw dataset is ~9,700 MB -> {9700/max(total_mb,1):.0f}x smaller)')
print(f'{len(manifest)} runs, {manifest.code.nunique()} codes, '
      f'{manifest.condition.nunique()} conditions, {manifest.fs_hz.iloc[0]} Hz')
print()
print(manifest.groupby(['damage_mode','fault_type']).size().to_string())